# Machine Learning Coursework: Telco Customer Churn Binary Classification
**Module**: Machine Learning — Individual Assignment
**Author**: Dinushi Hasangika
**Git Repository**: https://github.com/hasangikadinushi/telco-customer-churn-ml-

This notebook contains the complete implementation for:
- **Task 1**: Exploratory Data Analysis (EDA) & Visualizations (Figures 1, 2, 3)
- **Task 2**: Preprocessing, Modeling (KNN, Logistic Regression, Random Forest), 5-Fold CV Hyperparameter Tuning, and Evaluation (Figures 4, 5)
- **Task 3**: AI Ethics Audit (Demographic Fairness & Feature Importance) (Figures 6, 7)

## 1. Setup, Libraries & Global Styling

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report, roc_curve
)

os.makedirs('figures', exist_ok=True)
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'font.size': 11, 'figure.autolayout': True})
print('Libraries imported and styling initialized successfully!')

## 2. Load Dataset & Data Quality Inspection

In [ ]:
# Load the Telco Customer Churn dataset
# If running in Google Colab, ensure dataset is available
if not os.path.exists('Telco-Customer-Churn.csv'):
    import urllib.request
    url = 'https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv'
    urllib.request.urlretrieve(url, 'Telco-Customer-Churn.csv')
    print('Downloaded Telco-Customer-Churn.csv into runtime environment.')

df = pd.read_csv('Telco-Customer-Churn.csv')
print(f'Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns')

# Check for whitespace entries in TotalCharges (found 11 records with tenure=0)
whitespace_count = (df['TotalCharges'].str.strip() == '').sum()
print(f'Total whitespace entries found in TotalCharges: {whitespace_count}')

# Convert TotalCharges to float numeric and impute 0.0 for new customers
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce').fillna(0.0)
print('TotalCharges successfully converted to numeric float.')
df.head(3)

## 3. Task 1: Exploratory Data Analysis (EDA) & Visualizations

### Figure 1: Target Class Distribution (Customer Churn)

In [ ]:
# Figure 1: Target Class Distribution
plt.figure(figsize=(7, 5))
ax = sns.countplot(x='Churn', hue='Churn', data=df, palette=['#4C72B0', '#C44E52'], legend=False)
plt.title('Figure 1: Target Class Distribution (Customer Churn)', fontsize=13, weight='bold')
plt.xlabel('Churn Status', fontsize=11)
plt.ylabel('Number of Customers', fontsize=11)

for p in ax.patches:
    height = p.get_height()
    pct = (height / len(df)) * 100
    ax.annotate(f'{int(height)} ({pct:.1f}%)',
                (p.get_x() + p.get_width() / 2., height / 2),
                ha='center', va='center', color='white', weight='bold', fontsize=11)

plt.savefig('figures/fig1_churn_distribution.png', dpi=300)
plt.show()

### Figure 2: Impact of Contract Structure and Customer Tenure on Churn

In [ ]:
# Figure 2: Contract Type and Customer Tenure Density
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Subplot 1: Churn by Contract Type
sns.countplot(x='Contract', hue='Churn', data=df, ax=axes[0], palette=['#4C72B0', '#C44E52'])
axes[0].set_title('Churn Distribution by Contract Type', fontsize=12, weight='bold')
axes[0].set_xlabel('Contract Type', fontsize=11)
axes[0].set_ylabel('Number of Customers', fontsize=11)

# Subplot 2: Customer Tenure KDE Density
sns.kdeplot(data=df, x='tenure', hue='Churn', fill=True, common_norm=False,
            palette=['#4C72B0', '#C44E52'], ax=axes[1], alpha=0.5)
axes[1].set_title('Customer Tenure (Months) Density by Churn', fontsize=12, weight='bold')
axes[1].set_xlabel('Tenure (Months)', fontsize=11)
axes[1].set_ylabel('Density', fontsize=11)

plt.suptitle('Figure 2: Impact of Contract Structure and Customer Tenure on Churn', fontsize=14, weight='bold')
plt.savefig('figures/fig2_contract_and_tenure_churn.png', dpi=300)
plt.show()

### Figure 3: Financial and Service-Tier Drivers of Customer Churn

In [ ]:
# Figure 3: Monthly Charges Boxplot & Internet Service Churn Count
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Subplot 1: Monthly Charges vs Churn
sns.boxplot(x='Churn', y='MonthlyCharges', hue='Churn', data=df, ax=axes[0], palette=['#4C72B0', '#C44E52'], legend=False)
axes[0].set_title('Monthly Charges vs Churn Status', fontsize=12, weight='bold')
axes[0].set_xlabel('Churn Status', fontsize=11)
axes[0].set_ylabel('Monthly Charges ($)', fontsize=11)

# Subplot 2: Churn by Internet Service Type
sns.countplot(x='InternetService', hue='Churn', data=df, ax=axes[1], palette=['#4C72B0', '#C44E52'])
axes[1].set_title('Churn by Internet Service Type', fontsize=12, weight='bold')
axes[1].set_xlabel('Internet Service Type', fontsize=11)
axes[1].set_ylabel('Number of Customers', fontsize=11)

plt.suptitle('Figure 3: Financial and Service-Tier Drivers of Customer Churn', fontsize=14, weight='bold')
plt.savefig('figures/fig3_charges_and_internetservice.png', dpi=300)
plt.show()

## 4. Task 2: Data Preprocessing Pipeline

In [ ]:
# 1. Drop customer identifier
df_clean = df.drop(columns=['customerID'])

# 2. Binary mapping for target variable
df_clean['Churn'] = df_clean['Churn'].map({'No': 0, 'Yes': 1})

# 3. One-Hot Encoding for categorical features (drop_first=True avoids multicollinearity)
categorical_cols = df_clean.select_dtypes(include=['object']).columns.tolist()
df_encoded = pd.get_dummies(df_clean, columns=categorical_cols, drop_first=True, dtype=int)

# Separate features (X) and target (y)
X = df_encoded.drop(columns=['Churn'])
y = df_encoded['Churn']

# 4. Stratified Train-Test Split (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# 5. Feature Standardization using StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'Train shape: {X_train.shape}, Test shape: {X_test.shape}')
print(f'Train Churn Rate: {y_train.mean():.4f}, Test Churn Rate: {y_test.mean():.4f}')

## 5. Task 2: Model Training & Evaluation
We evaluate **KNN**, **Logistic Regression**, and **Random Forest** across two settings:
- **Setting 1**: Baseline models (default parameters)
- **Setting 2**: Tuned models (via 5-Fold Stratified Cross-Validation with GridSearchCV)

In [ ]:
results = []
trained_models = {}

def record_metrics(model_name, setting, y_true, y_pred, y_prob):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    auc = roc_auc_score(y_true, y_prob)
    results.append({
        'Model': model_name, 'Setting': setting,
        'Accuracy': acc, 'Precision': prec, 'Recall': rec,
        'F1-Score': f1, 'ROC-AUC': auc
    })
    print(f'[{setting:8s}] {model_name:20s} | Acc: {acc:.4f} | Prec: {prec:.4f} | Rec: {rec:.4f} | F1: {f1:.4f} | AUC: {auc:.4f}')

# --- EXPERIMENTAL SETTING 1: BASELINE MODELS ---
print('=== EXPERIMENTAL SETTING 1: BASELINE MODELS ===')
# Baseline KNN
knn_base = KNeighborsClassifier().fit(X_train_scaled, y_train)
trained_models['KNN_Baseline'] = (knn_base, knn_base.predict(X_test_scaled), knn_base.predict_proba(X_test_scaled)[:, 1])
record_metrics('KNN', 'Baseline', y_test, trained_models['KNN_Baseline'][1], trained_models['KNN_Baseline'][2])

# Baseline Logistic Regression
lr_base = LogisticRegression(random_state=42, max_iter=1000).fit(X_train_scaled, y_train)
trained_models['LR_Baseline'] = (lr_base, lr_base.predict(X_test_scaled), lr_base.predict_proba(X_test_scaled)[:, 1])
record_metrics('Logistic Regression', 'Baseline', y_test, trained_models['LR_Baseline'][1], trained_models['LR_Baseline'][2])

# Baseline Random Forest
rf_base = RandomForestClassifier(random_state=42).fit(X_train, y_train)
trained_models['RF_Baseline'] = (rf_base, rf_base.predict(X_test), rf_base.predict_proba(X_test)[:, 1])
record_metrics('Random Forest', 'Baseline', y_test, trained_models['RF_Baseline'][1], trained_models['RF_Baseline'][2])

In [ ]:
# --- EXPERIMENTAL SETTING 2: HYPERPARAMETER TUNED MODELS ---
print('=== EXPERIMENTAL SETTING 2: 5-FOLD CV HYPERPARAMETER TUNING ===')
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# 1. Tuned KNN
grid_knn = GridSearchCV(KNeighborsClassifier(), {
    'n_neighbors': [5, 9, 13, 17, 21],
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan']
}, cv=cv, scoring='roc_auc', n_jobs=-1).fit(X_train_scaled, y_train)
knn_tuned = grid_knn.best_estimator_
trained_models['KNN_Tuned'] = (knn_tuned, knn_tuned.predict(X_test_scaled), knn_tuned.predict_proba(X_test_scaled)[:, 1])
record_metrics('KNN', 'Tuned', y_test, trained_models['KNN_Tuned'][1], trained_models['KNN_Tuned'][2])
print(f'  Best KNN Params: {grid_knn.best_params_}')

# 2. Tuned Logistic Regression
grid_lr = GridSearchCV(LogisticRegression(random_state=42, max_iter=1000), {
    'C': [0.01, 0.1, 1.0, 10.0],
    'solver': ['liblinear', 'lbfgs'],
    'class_weight': [None, 'balanced']
}, cv=cv, scoring='roc_auc', n_jobs=-1).fit(X_train_scaled, y_train)
lr_tuned = grid_lr.best_estimator_
trained_models['LR_Tuned'] = (lr_tuned, lr_tuned.predict(X_test_scaled), lr_tuned.predict_proba(X_test_scaled)[:, 1])
record_metrics('Logistic Regression', 'Tuned', y_test, trained_models['LR_Tuned'][1], trained_models['LR_Tuned'][2])
print(f'  Best LR Params: {grid_lr.best_params_}')

# 3. Tuned Random Forest
grid_rf = GridSearchCV(RandomForestClassifier(random_state=42), {
    'n_estimators': [100, 200],
    'max_depth': [6, 10, None],
    'min_samples_split': [2, 5],
    'class_weight': [None, 'balanced']
}, cv=cv, scoring='roc_auc', n_jobs=-1).fit(X_train, y_train)
rf_tuned = grid_rf.best_estimator_
trained_models['RF_Tuned'] = (rf_tuned, rf_tuned.predict(X_test), rf_tuned.predict_proba(X_test)[:, 1])
record_metrics('Random Forest', 'Tuned', y_test, trained_models['RF_Tuned'][1], trained_models['RF_Tuned'][2])
print(f'  Best RF Params: {grid_rf.best_params_}')

### Quantitative Benchmark Summary Table

In [ ]:
results_df = pd.DataFrame(results)
results_df.to_csv('model_comparison_results.csv', index=False)
display(results_df)

### Figure 4: Confusion Matrix Comparison Across Models & Settings

In [ ]:
# Figure 4: Confusion Matrices
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
models_to_plot = [
    ('KNN (Baseline)', 'KNN_Baseline'),
    ('Logistic Reg (Baseline)', 'LR_Baseline'),
    ('Random Forest (Baseline)', 'RF_Baseline'),
    ('KNN (Tuned)', 'KNN_Tuned'),
    ('Logistic Reg (Tuned)', 'LR_Tuned'),
    ('Random Forest (Tuned)', 'RF_Tuned')
]

for idx, (title, key) in enumerate(models_to_plot):
    row, col = idx // 3, idx % 3
    _, y_pred, _ = trained_models[key]
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=axes[row, col],
                annot_kws={'size': 12, 'weight': 'bold'})
    axes[row, col].set_title(title, fontsize=12, weight='bold')
    axes[row, col].set_xlabel('Predicted Label', fontsize=10)
    axes[row, col].set_ylabel('Actual Label', fontsize=10)
    axes[row, col].set_xticklabels(['No Churn (0)', 'Churn (1)'])
    axes[row, col].set_yticklabels(['No Churn (0)', 'Churn (1)'])

plt.suptitle('Figure 4: Confusion Matrix Comparison Across Models & Settings', fontsize=14, weight='bold')
plt.savefig('figures/fig4_confusion_matrices.png', dpi=300)
plt.show()

### Figure 5: Receiver Operating Characteristic (ROC) Curves

In [ ]:
# Figure 5: ROC Curves Comparison
plt.figure(figsize=(9, 6))
colors = {
    'KNN_Baseline': ('#55A868', '--', 'KNN (Baseline)'),
    'LR_Baseline': ('#4C72B0', '--', 'Logistic Regression (Baseline)'),
    'RF_Baseline': ('#C44E52', '--', 'Random Forest (Baseline)'),
    'KNN_Tuned': ('#2ca02c', '-', 'KNN (Tuned)'),
    'LR_Tuned': ('#1f77b4', '-', 'Logistic Regression (Tuned)'),
    'RF_Tuned': ('#d62728', '-', 'Random Forest (Tuned)')
}

for key, (color, linestyle, label) in colors.items():
    _, _, y_prob = trained_models[key]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc_val = roc_auc_score(y_test, y_prob)
    plt.plot(fpr, tpr, label=f'{label} (AUC = {auc_val:.3f})', color=color, linestyle=linestyle, lw=2)

plt.plot([0, 1], [0, 1], 'k:', lw=1.5, label='Random Chance (AUC = 0.500)')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate (1 - Specificity)', fontsize=11)
plt.ylabel('True Positive Rate (Recall / Sensitivity)', fontsize=11)
plt.title('Figure 5: Receiver Operating Characteristic (ROC) Curves', fontsize=13, weight='bold')
plt.legend(loc='lower right', fontsize=10)
plt.savefig('figures/fig5_roc_curves.png', dpi=300)
plt.show()

## 6. Task 3: AI Ethics, Explainability & Post-Deployment Audit

### Figure 6: Top 10 Most Important Features (Random Forest Tuned)

In [ ]:
# Figure 6: Feature Importances
rf_best_model = trained_models['RF_Tuned'][0]
importances = rf_best_model.feature_importances_
feature_imp_df = pd.DataFrame({'Feature': X.columns, 'Importance': importances}).sort_values('Importance', ascending=False).head(10)

plt.figure(figsize=(10, 5))
sns.barplot(x='Importance', y='Feature', hue='Feature', data=feature_imp_df, palette='viridis', legend=False)
plt.title('Figure 6: Top 10 Most Important Features (Random Forest Tuned)', fontsize=13, weight='bold')
plt.xlabel('Mean Decrease in Impurity (Feature Importance)', fontsize=11)
plt.ylabel('Feature Name', fontsize=11)
plt.savefig('figures/fig6_feature_importance.png', dpi=300)
plt.show()

### Figure 7: AI Ethics & Fairness Audit across Demographic Subgroups

In [ ]:
# Figure 7: Demographic Fairness Audit (SeniorCitizen and gender)
test_subgroup = df.loc[X_test.index].copy()
test_subgroup['Actual_Churn'] = y_test
test_subgroup['Pred_Churn'] = trained_models['RF_Tuned'][1]

senior_stats = test_subgroup.groupby('SeniorCitizen').agg(
    Actual_Churn_Rate=('Actual_Churn', 'mean'),
    Predicted_Churn_Rate=('Pred_Churn', 'mean')
).reset_index()
senior_stats['SeniorCitizen'] = senior_stats['SeniorCitizen'].map({0: 'Non-Senior Citizen', 1: 'Senior Citizen'})

gender_stats = test_subgroup.groupby('gender').agg(
    Actual_Churn_Rate=('Actual_Churn', 'mean'),
    Predicted_Churn_Rate=('Pred_Churn', 'mean')
).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Subplot 1: Senior Citizen Audit
senior_melted = senior_stats.melt(id_vars='SeniorCitizen',
                                  value_vars=['Actual_Churn_Rate', 'Predicted_Churn_Rate'],
                                  var_name='Metric', value_name='Rate')
sns.barplot(x='SeniorCitizen', y='Rate', hue='Metric', data=senior_melted, ax=axes[0], palette=['#4C72B0', '#C44E52'])
axes[0].set_title('Actual vs Predicted Churn by Senior Citizen Status', fontsize=12, weight='bold')
axes[0].set_ylabel('Churn Proportion', fontsize=11)
axes[0].set_ylim(0, 0.7)

# Subplot 2: Gender Audit
gender_melted = gender_stats.melt(id_vars='gender',
                                  value_vars=['Actual_Churn_Rate', 'Predicted_Churn_Rate'],
                                  var_name='Metric', value_name='Rate')
sns.barplot(x='gender', y='Rate', hue='Metric', data=gender_melted, ax=axes[1], palette=['#4C72B0', '#C44E52'])
axes[1].set_title('Actual vs Predicted Churn by Gender', fontsize=12, weight='bold')
axes[1].set_ylabel('Churn Proportion', fontsize=11)
axes[1].set_ylim(0, 0.7)

plt.suptitle('Figure 7: AI Ethics & Fairness Audit across Demographic Subgroups', fontsize=14, weight='bold')
plt.savefig('figures/fig7_ethics_fairness_audit.png', dpi=300)
plt.show()

print('Demographic Fairness Tables:')
print(senior_stats)
print(gender_stats)